## Milestone 3 II4035

By: Samuel Eric Yonatan - 18221133

Subtopic: fraud detection in mobile money transaction app

Data source: https://www.kaggle.com/datasets/ealaxi/paysim1?select=PS_20174392719_1491204439457_log.csv

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Reading a CSV file containing a dataset
df = pd.read_csv('./mobile_money_transaction.csv')
df.head()

## Data Description

In [ ]:
# The dataset has 6,362,620 rows and 11 columns.
df.shape

In [ ]:
# The dataset contains data types such as integer, string (object), and float.
df.info(verbose=True)

In [ ]:
# There is a typo in the attribute name, which should be "oldbalanceOrig" instead of "oldbalanceOrg."

df.rename(columns={'oldbalanceOrg': 'oldbalanceOrig'}, inplace=True)
df.columns

# The attribute "oldbalanceOrg" has been successfully changed to "oldbalanceOrig."

Dataset columns
- step: the time elapsed for the transaction, in hours
- type: transaction type (cash out, cash in, debit, payment, transfer)
- amount: the amount of money involved in the transaction
- nameOrig: the customer who initiated the transaction
- oldbalanceOrig: the customer's balance before the transaction
- newbalanceOrig: the customer's balance after the transaction
- nameDest: the recipient of the transaction
- oldbalanceDest: the recipient's balance before the transaction
- newbalanceDest: the recipient's balance after the transaction
- isFraud: identification of fraudulent transactions (1 if fraud, 0 if not fraud)
- isFlaggedFraud: system identification of fraud in the transaction

In [ ]:
df[['step', 'amount', 'oldbalanceOrig', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']].describe()

# The attributes "isFraud" and "isFlaggedFraud" do not need to be subjected to statistical analysis because they only have values of 0 and 1, which represent non-fraudulent and fraudulent transactions, respectively.

In [ ]:
# For the transaction types, there are "cash out," "payment," "cash in," "transfer," and "debit."
df['type'].value_counts()

## Data Quality Verification

In [ ]:
# To identify missing data
df.isnull().sum()

# There is no missing or null data in the dataset.

In [ ]:
# To identify the presence of missing data (if missing data is represented by '?'):
for i in range(len(df.columns)):
  print('Attribute-', i + 1, ': ', df.columns[i], ', Number of ?: ', df[df[df.columns[i]] == '?'].shape[0])

# There is no missing or null data in the dataset.

In [ ]:
# To identify imbalanced data based on the "isFraud" attribute:
print('No Frauds: ', round(df['isFraud'].value_counts()[0]/len(df) * 100, 2), '% of the dataset')
print('Frauds: ', round(df['isFraud'].value_counts()[1]/len(df) * 100, 2), '% of the dataset')

# The dataset used is imbalanced, with a much larger number of non-fraudulent transactions compared to fraudulent transactions.

In [ ]:
# Outliers identification
for i in (df.columns):
    if (df[i].dtypes in ['int64','float64']):
        print('\nAttribute-',i,':',df[i].dtypes)
        Q1=df[i].quantile(0.25)
        print('Q1',Q1)
        Q3=df[i].quantile(0.75)
        print('Q3',Q3)
        IQR=Q3-Q1
        print('IQR',IQR)
        min=df[i].min()
        max=df[i].max()
        min_IQR=Q1-1.5*IQR
        max_IQR=Q3+1.5*IQR
        if (min<min_IQR):
            print('Low outlier is found')
        if (max>max_IQR):
            print('High outlier is found')


## Data Exploration

In [ ]:
# Identify transactions that are fraudulent but are not classified as fraud by the system
df[(df['isFraud'] == 1) & (df['isFlaggedFraud'] == 0)]

# There are 8197 data points that are fraudulent but are not flagged as fraud by the system.

In [ ]:
# Identify the number of fraudulent transactions based on each transaction type:
df.groupby('type')['isFraud'].sum()

# Only the transaction types "cash out" and "transfer" have fraudulent transactions.

In [ ]:
# Identify transactions with an amount less than or equal to 0:
print('Number of transactions with amount < 0: ', len(df[df['amount'] < 0]))
print('Number of transactions amount = 0: ', len(df[df['amount'] == 0]))

# There are 16 transactions with an amount equal to 0.

In [ ]:
# Identify whether all transactions with an amount equal to 0 are considered fraud:
df[df['amount'] == 0]['isFraud'].value_counts()[1] == len(df[df['amount'] == 0])

# All transactions with an amount equal to 0 are considered fraud.

In [ ]:
sns.countplot(data=df, x='type')
plt.title('Number of Transactions for Every Type')

# From the graph below, it can be seen that the transaction types with the most transactions are "payment" and "cash out."

In [ ]:
df_fraud = pd.DataFrame({
  'isFraud': ['Not Fraud', 'Fraud'],
  'count': [len(df) - df.groupby('isFraud')['isFraud'].sum().values[1], df.groupby('isFraud')['isFraud'].sum().values[1]]
})
# df_fraud
plt.pie(df_fraud['count'], labels=df_fraud['isFraud'], autopct='%1.1f%%')
plt.title('Comparison between Number of Fraudulent dan Non-fraudulent Transactions')
plt.show()

# From the graph below, it can be seen that the number of fraudulent transactions is very small compared to the number of non-fraudulent transactions.

In [ ]:
df_fraud_type = df[df['isFraud'] == 1]
sns.countplot(data=df_fraud_type, x='type')
plt.title('Comparison between Number of Fraudulent Transactions per Type')

# From the graph below, the number of fraudulent transactions is roughly evenly split between two types, namely transfer and cash out.

In [ ]:
df_temp = df.select_dtypes(include=['int64', 'float64'])
corr = df_temp.corr()

sns.set(rc={'figure.figsize': (25, 15)})
sns.heatmap(corr, annot=True, cmap='RdYlGn', fmt='.2f')
plt.title('Correlation between Numerical Attribute')
plt.show()

# From the graph below, it can be seen that "oldbalanceDest" and "newbalanceDest" are correlated, as well as "oldbalanceOrig" and "newbalanceOrig" which are also correlated.

## Next Steps

Here are the next steps that can be taken:

1. Remove some attributes because there are some attributes that are correlated.
2. Sample the non-fraudulent transactions to balance the number of fraudulent and non-fraudulent transactions.

## Data Preparation

### Cleansing Data

There are no data with empty values, and all data already have the correct data types.

### Data Formatting

All attributes in the dataset do not need to be encoded because they already have the correct data types.

In [ ]:
# Since only transfer and cash-out transactions have fraud transactions, other transaction types that are not transfer and cash-out are mapped to other types.

type_map = {
  'PAYMENT':'OTHERS',
  'TRANSFER':'TRANSFER',
  'CASH_OUT':'CASH_OUT',
  'DEBIT':'OTHERS',
  'CASH_IN':'OTHERS'
}

df['type'] = df['type'].map(type_map)

In [ ]:
# Perform one-hot encoding on a transaction type attribute to make it numeric

from sklearn.preprocessing import OneHotEncoder

oh_enc = OneHotEncoder()
results = pd.DataFrame(oh_enc.fit_transform(df[['type']]).toarray(), columns=['type_cash_out', 'type_others', 'type_transfer'])
df = pd.concat([df, results], axis=1)
df

### Handling Imbalanced Data

In [ ]:
# Since non-fraudulent transactions outnumber fraudulent transactions, undersampling will be performed on non-fraudulent transactions to balance the number of fraudulent and non-fraudulent transactions.

class_fraud = df[df['isFraud'] == 1]
class_not_fraud = df[df['isFraud'] == 0]

class_not_fraud_under = class_not_fraud.sample(class_fraud.shape[0])
new_df = pd.concat([class_not_fraud_under, class_fraud], axis=0)
new_df['isFraud'].value_counts()

### Feature Engineering

#### Feature Transformation/Generation

In [ ]:
# Creating a new attribute that describes the balance inaccuracy between origin, destination, and amount.

new_df['balanceOrig_inacc'] = ((new_df['oldbalanceOrig'] - new_df['amount']) != new_df['newbalanceOrig']).astype(int)
new_df['balanceDest_inacc'] = ((new_df['oldbalanceDest'] + new_df['amount']) != new_df['newbalanceDest']).astype(int)
new_df.head()

#### Feature Selection

In [ ]:
# Checking the correlation of each attribute to select attributes that can be good features for the model to be built.

corr = new_df.select_dtypes(['int64', 'float64']).corr()
corr

In [ ]:
# The selected attributes to become features are those that are correlated with the isFraud attribute and not correlated with other attributes. These attributes are type_cash_out, type_others, type_transfer, step, amount, oldbalanceOrig, newbalanceDest, balanceDest_inacc, balanceOrig_inacc, and isFraud as the attribute to be predicted.

new_df = new_df[['type_cash_out', 'type_others', 'type_transfer', 'step', 'amount', 'oldbalanceOrig', 'newbalanceDest', 'balanceDest_inacc', 'balanceOrig_inacc', 'isFraud']]
new_df

## Modelling

### Data Split

In [ ]:
# Data is split into testing data and training data, with the training data size being 70% of the total data.

from sklearn.model_selection import train_test_split

X = new_df.drop('isFraud', axis=1)
y = new_df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7)

print('Number of training data: ', len(X_train))
print('Number of testing data: ', len(X_test))

In [ ]:
# Import the required library to measure the performance of the machine learning model created.

from sklearn import metrics

### k-NN Classifier

In [ ]:
# Using the k-NN classifier method to predict the classification of the isFraud attribute

from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
score = metrics.accuracy_score(y_test, y_pred)
print('Accuracy: ', score)

In [ ]:
print('Classification Report: \n\n', metrics.classification_report(y_test, y_pred))

### Decision Tree Classifier

In [ ]:
# Using the decision tree classifier method to predict the classification of the isFraud attribute

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
  max_depth=None,
  min_samples_split=2
)

dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
score = metrics.accuracy_score(y_test, y_pred)
print('Accuracy: ', score)

In [ ]:
print('Classification Report: \n\n', metrics.classification_report(y_test, y_pred))

### Naive Bayes Classifier

In [ ]:
# Using the naive Bayes classifier method to predict the classification of the isFraud attribute

from sklearn import naive_bayes

nb = naive_bayes.BernoulliNB()

nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)
score = metrics.accuracy_score(y_test, y_pred)
print('Accuracy: ', score)

In [ ]:
print('Classification Report: \n\n', metrics.classification_report(y_test, y_pred))

**Conclusion**: The best accuracy and F1 score were achieved using the Decision Tree Classifier method, with an accuracy and F1 score of 0.99.